# 03 · Procurement EDA
Exploratory analysis of the cleaned dataset covering the mandated questions: price by component & country, macro co-movement, economies of scale, supplier delivery/quality/risk.

In [ ]:
%matplotlib inline
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40, "display.width", 160)
from src import config as C
print("project root:", ROOT)

In [ ]:
df = pd.read_csv(C.PROCESSED_QUOTES_CSV, parse_dates=['date'])
print(df.shape); df.head()

## Price by component type and by country

In [ ]:
display(df.groupby('component_type').unit_price.median().sort_values(ascending=False).round(0).to_frame('median_price'))
display(df.groupby('country').unit_price.median().sort_values(ascending=False).round(0).to_frame('median_price'))

## Macro co-movement: monthly average price vs commodity index

In [ ]:
m = df.groupby('date').agg(avg_price=('unit_price','mean'), rawmat=('raw_material_index','mean'),
                           energy=('energy_cost_index','mean')).reset_index()
fig, ax = plt.subplots(figsize=(11,5))
ax.plot(m.date, m.avg_price/m.avg_price.iloc[0]*100, lw=3, color='#b3122b', label='Avg unit price (idx)')
ax.plot(m.date, m.rawmat, lw=2, ls='--', label='Raw-material index')
ax.plot(m.date, m.energy, lw=2, ls=':', label='Energy index')
ax.set_title('Monthly average price vs real macro indices (rebased)'); ax.legend(); plt.show()

## Economies of scale & macro sensitivity

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(14,5))
sns.regplot(data=df, x='order_volume', y='price_ratio_vs_median', logx=True, ax=axes[0],
            scatter_kws=dict(alpha=.15,s=12,color='gray'), line_kws=dict(color='#b3122b'))
axes[0].set_xscale('log'); axes[0].axhline(1,color='k',lw=.7,alpha=.4); axes[0].set_title('Order volume vs price level')
ac = df[df.component_type=='Aluminum Casting']
sns.regplot(data=ac, x='raw_material_index', y='unit_price', ax=axes[1],
            scatter_kws=dict(alpha=.25,s=18,color='gray'), line_kws=dict(color='#b3122b'))
axes[1].set_title('Raw material vs price (Aluminum Casting)'); plt.tight_layout(); plt.show()

## Supplier reliability, quality and risk

In [ ]:
sup = df.groupby(['supplier_id','country']).agg(
    otd=('on_time_delivery_rate','mean'), defect=('defect_rate','mean'),
    lead=('lead_time_days','mean'), risk=('risk_score_external','mean')).reset_index()
print('Risk by region:'); display(df.groupby('region').risk_score_external.mean().round(1).to_frame())
fig, ax = plt.subplots(figsize=(9,6))
sns.scatterplot(data=sup, x='otd', y='defect', hue='country', s=70, ax=ax)
ax.set_title('Suppliers: on-time delivery vs defect rate'); ax.invert_yaxis(); plt.show()

**Read-out.** Average price tracks the real commodity & energy cycle; larger orders price below the component median; overseas regions (Asia, the Americas) carry higher risk.